# Reto: agentes inteligentes en un laberinto

Este notebook presenta un generador basado en Kruskal y compara una búsqueda BFS determinista con un explorador estocástico por fronteras. La implementación se importa desde el paquete `laberinto` para mantener una sola fuente de verdad.

In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from laberinto import AgenteBusquedaDeterminista, AgenteExploradorFronteras, Entorno, crear_laberinto
from laberinto.visualizacion import dibujar_laberinto

## 1. Generación del entorno

El generador construye entre uno y tres componentes mediante Kruskal aleatorio. Después agrega hasta cuatro conexiones internas. No busca caminos ni comprueba si la salida es alcanzable.

In [ ]:
FILAS, COLUMNAS, SEMILLA = 15, 15, 26
laberinto, entrada, salida = crear_laberinto(FILAS, COLUMNAS, SEMILLA)
print(f'Entrada: {entrada} | Salida: {salida}')
dibujar_laberinto(laberinto, entrada, salida)

## 2. Búsqueda determinista en anchura (BFS)

BFS expande cada celda accesible como máximo una vez y utiliza una cola FIFO. Como todos los movimientos cuestan lo mismo, la primera salida encontrada produce un camino mínimo.

In [ ]:
determinista = AgenteBusquedaDeterminista(Entorno(laberinto))
informe_determinista, eventos_determinista = determinista.ejecutar()
informe_determinista

In [ ]:
evento_final_determinista = eventos_determinista[-1]
dibujar_laberinto(laberinto, entrada, salida, evento_final_determinista, solo_ruta=True)

## 3. Exploración estocástica por fronteras

El agente solo percibe su celda actual. A* no intenta adivinar la salida: calcula rutas hacia fronteras ya descubiertas dentro del mapa conocido.

In [ ]:
explorador = AgenteExploradorFronteras(Entorno(laberinto), semilla=SEMILLA)
informe_fronteras, eventos_fronteras = explorador.ejecutar()
informe_fronteras

In [ ]:
evento_final = eventos_fronteras[-1]
dibujar_laberinto(laberinto, entrada, salida, evento_final, vista_agente=True)

## 4. Comparación

BFS evita expansiones repetidas y garantiza una ruta mínima porque todos los movimientos tienen costo unitario. La exploración por fronteras representa mejor un agente situado: construye memoria, se desplaza físicamente y desconoce la salida. Su trayectoria no tiene por qué ser la ruta más corta, pero la ruta final siempre utiliza conexiones descubiertas.

In [ ]:
comparacion = {
    'BFS determinista': {
        'solucion': informe_determinista['tiene_solucion'],
        'expandidas': informe_determinista['expandidas'],
        'descubiertas': informe_determinista['celdas_descubiertas'],
        'longitud_ruta': len(informe_determinista['camino'] or []),
    },
    'Fronteras': {
        'solucion': informe_fronteras['tiene_solucion'],
        'expandidas': informe_fronteras['expandidas'],
        'movimientos': informe_fronteras['movimientos'],
        'longitud_ruta': len(informe_fronteras['camino'] or []),
    },
}
comparacion